In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True

print("Done")


In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd

data_path = os.path.join(path, "Q3_data.csv")
print (data_path)

df = pd.read_csv(data_path)


In [ ]:
print("First 5 rows of the dataset:")
display(df.head())



In [ ]:
print("\nData info:")
print(df.info())



In [ ]:
print("\nStatistical summary for numeric columns:")
display(df.describe())

In [ ]:
!pip install catboost

In [ ]:
from catboost import CatBoostClassifier


In [ ]:
# Task 1: Write your code here:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

df_model = df.copy()

print("Missing values per column (first 10):")
print(df_model.isna().sum().head(10))
print("Total missing values in dataset:", df_model.isna().sum().sum())

df_model = df_model.dropna(subset=["Target"])
print("Shape after ensuring non-missing target:", df_model.shape)



In [ ]:
# Task 2: Write your code here:
print("Shape before dropping duplicates:", df_model.shape)
df_model = df_model.drop_duplicates()
print("Shape after dropping duplicates:", df_model.shape)


In [ ]:
# Task 3: Write your code here:
categorical_features = df_model.select_dtypes(include=["object"]).columns.tolist()
print("Categorical features detected:", categorical_features)




In [ ]:
# Task 4: Write your code here:
X = df_model.drop(columns=["Target"])
y = df_model["Target"].astype(int)

numeric_features = X.columns.tolist()
print("Number of numerical features:", len(numeric_features))

model_full = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", CatBoostClassifier(
        depth=6,
        learning_rate=0.1,
        n_estimators=300,
        loss_function="Logloss",
        eval_metric="F1",
        verbose=False,
        random_state=42
    ))
])


In [ ]:
# Task 5: Write your code here:
class_counts = y.value_counts()
class_ratios = y.value_counts(normalize=True)

print("Target class counts:")
print(class_counts)
print("Target class ratios:")
print(class_ratios)

imbalance_ratio = class_ratios.max()
if imbalance_ratio > 0.6:
    print("The target is imbalanced. F1-score will be used as the main evaluation metric.")
else:
    print("The target is relatively balanced. F1-score will still be used as the main evaluation metric.")


In [ ]:
# Task 1: Write your code here:

print("Final X shape:", X.shape)
print("Final y shape:", y.shape)

In [ ]:
# Task 2,3,4,5: Write your code here:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores_full = []
fold_index = 1


for train_idx, test_idx in skf.split(X, y):
    print(f"Starting fold {fold_index}")
    X_train, X_valid = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[test_idx]

    model_full.fit(X_train, y_train)
    y_pred = model_full.predict(X_valid)

    f1 = f1_score(y_valid, y_pred)
    f1_scores_full.append(f1)
    print(f"Fold {fold_index} F1-score: {f1:.4f}")
    fold_index += 1

avg_f1_full = np.mean(f1_scores_full)
print("F1-scores for each fold (full model):", f1_scores_full)
print(f"Average F1-score across folds (full model): {avg_f1_full:.4f}")


In [ ]:
model_full.fit(X, y)
cat_model = model_full.named_steps["classifier"]
feature_importances = cat_model.get_feature_importance()
feature_names = X.columns.tolist()

sorted_idx = np.argsort(feature_importances)[::-1]

plt.figure(figsize=(10, 8))
top_n = 20 if len(feature_names) > 20 else len(feature_names)
top_idx = sorted_idx[:top_n]
plt.barh(range(top_n), feature_importances[top_idx][::-1])
plt.yticks(range(top_n), [feature_names[i] for i in top_idx][::-1])
plt.xlabel("Feature importance")
plt.title("Top feature importances (CatBoostClassifier)")
plt.tight_layout()
plt.show()

In [ ]:
golden_index = sorted_idx[0]
golden_feature = feature_names[golden_index]
golden_importance = feature_importances[golden_index]

print("Most important feature (golden feature):", golden_feature)
print("Importance of golden feature:", golden_importance)


In [ ]:
X_golden = df_model[[golden_feature]]
print("Shape of X with only golden feature:", X_golden.shape)

model_golden = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", CatBoostClassifier(
        depth=6,
        learning_rate=0.1,
        n_estimators=300,
        loss_function="Logloss",
        eval_metric="F1",
        verbose=False,
        random_state=42
    ))
])

f1_scores_golden = []
fold_index = 1

for train_idx, test_idx in skf.split(X_golden, y):
    print(f"Starting fold (golden feature) {fold_index}")
    X_train_g, X_valid_g = X_golden.iloc[train_idx], X_golden.iloc[test_idx]
    y_train_g, y_valid_g = y.iloc[train_idx], y.iloc[test_idx]

    model_golden.fit(X_train_g, y_train_g)
    y_pred_g = model_golden.predict(X_valid_g)

    f1_g = f1_score(y_valid_g, y_pred_g)
    f1_scores_golden.append(f1_g)
    print(f"Fold {fold_index} F1-score (golden feature): {f1_g:.4f}")
    fold_index += 1

avg_f1_golden = np.mean(f1_scores_golden)
print("F1-scores for each fold (golden feature model):", f1_scores_golden)
print(f"Average F1-score across folds (golden feature model): {avg_f1_golden:.4f}")

print(f"Full model average F1-score: {avg_f1_full:.4f}")
print(f"Golden feature model average F1-score: {avg_f1_golden:.4f}")